# Nanochat D8 Quant Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and exports quantized artifacts:

- `int8_linear` with default embedding behavior
- `fp16_copy`
- AWQ-style `awq_int4`

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` available, one GPU is enough for serving/eval)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes quant artifacts under `chatquant_exports`.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/temp/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working', '/kaggle/temp'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


Code input: /kaggle/input/datasets/yshuaiqin/nanochat-codes
SFT cache input: /kaggle/input/datasets/yshuaiqin/nanochat-d8-sft-cache
Working repo: /kaggle/working/nanochat
Nanochat cache: /kaggle/working/nanochat_cache
HF cache: /kaggle/temp/huggingface_cache
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
overlay         7.9T  6.8T  1.1T  87% /
2.1G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Install Dependencies

Install the runtime packages used by quant export, eval, and the quantized web server.


In [2]:
packages = [
    'datasets>=4.0.0',
    'fastapi>=0.115.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'pydantic>=2.0.0',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'uvicorn>=0.30.0',
    'zstandard>=0.25.0',
]
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade-strategy',
    'only-if-needed',
] + packages)
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.8 MB/s eta 0:00:00
Dependencies installed


## Configure Runtime

Keep compilation disabled and use float32 while exporting/evaluating quantized artifacts.


In [3]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
env['HF_HUB_DISABLE_XET'] = '1'
env['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
env['HF_HUB_ETAG_TIMEOUT'] = '60'
env['TMPDIR'] = '/kaggle/temp'
env['PYTHONUNBUFFERED'] = '1'
env['NANOCHAT_DTYPE'] = 'float32'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'
env['OMP_NUM_THREADS'] = '1'

os.environ.update(env)

def run_cmd(cmd, **kwargs):
    cmd = [str(x) for x in cmd]
    print('\\n$', ' '.join(cmd))
    return subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True, **kwargs)

def latest_model_step(checkpoint_dir):
    files = sorted(Path(checkpoint_dir).glob('model_*.pt'))
    assert files, f'No model_*.pt files found in {checkpoint_dir}'
    return max(int(path.stem.split('_')[-1]) for path in files)

def latest_quant_artifact(export_dir):
    files = sorted(Path(export_dir).glob('quant_*.pt'))
    assert files, f'No quant_*.pt files found in {export_dir}'
    return files[-1]

print('NANOCHAT_BASE_DIR:', env['NANOCHAT_BASE_DIR'])
print('HF_HOME:', env['HF_HOME'])
print('NANOCHAT_DTYPE:', env['NANOCHAT_DTYPE'])


NANOCHAT_BASE_DIR: /kaggle/working/nanochat_cache
HF_HOME: /kaggle/temp/huggingface_cache
NANOCHAT_DTYPE: float32


In [4]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

MODEL_TAG = 'd8_kaggle'
SFT_DIR = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
SFT_STEP = latest_model_step(SFT_DIR)
print('SFT checkpoint dir:', SFT_DIR)
print('Latest SFT step:', SFT_STEP)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
SFT checkpoint dir: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle
Latest SFT step: 4999


## Validate Repo And SFT Cache

Check the active Kaggle working copy and attached SFT cache before exporting artifacts.


In [5]:
quant_scripts = [
    'scripts/chat_quantize.py',
    'scripts/chat_quant_awq.py',
    'scripts/chat_quant_eval.py',
    'scripts/chat_web_quant.py',
]

for rel_path in quant_scripts:
    assert (WORK_REPO / rel_path).exists(), f'Missing {rel_path}'
assert (WORK_CACHE / 'tokenizer').exists(), 'Missing tokenizer directory'
assert SFT_DIR.exists(), f'Missing SFT checkpoint dir: {SFT_DIR}'
assert sorted(SFT_DIR.glob('model_*.pt')), f'No model_*.pt files found in {SFT_DIR}'
assert sorted(SFT_DIR.glob('meta_*.json')), f'No meta_*.json files found in {SFT_DIR}'

run_cmd([sys.executable, '-m', 'py_compile'] + quant_scripts)

for module in [
    'scripts.chat_quantize',
    'scripts.chat_quant_awq',
    'scripts.chat_quant_eval',
    'scripts.chat_web_quant',
]:
    result = subprocess.run(
        [sys.executable, '-m', module, '--help'],
        cwd=WORK_REPO,
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    print('\\nHELP', module)
    print('\\n'.join(result.stdout.splitlines()[:18]))


\n$ /usr/bin/python3 -m py_compile scripts/chat_quantize.py scripts/chat_quant_awq.py scripts/chat_quant_eval.py scripts/chat_web_quant.py
\nHELP scripts.chat_quantize
usage: chat_quantize.py [-h] [--source {base,sft,rl}]\n                        [--checkpoint-dir CHECKPOINT_DIR]\n                        [--model-tag MODEL_TAG] [--step STEP]\n                        [--method {int8_sym,int8_linear,fp16_copy}]\n                        [--quantize-embeddings [QUANTIZE_EMBEDDINGS]]\n                        [--output-dir OUTPUT_DIR] [--suffix SUFFIX]\n\nQuantize a nanochat checkpoint into a separate export artifact\n\noptions:\n  -h, --help            show this help message and exit\n  --source {base,sft,rl}\n                        Built-in checkpoint family\n  --checkpoint-dir CHECKPOINT_DIR\n                        Custom checkpoints root directory\n  --model-tag MODEL_TAG\n                        Model tag inside the checkpoint root\n  --step STEP           Checkpoint step
\nHELP scrip

## Export INT8 And FP16 Artifacts

Export `int8_linear` and `fp16_copy` artifacts from the SFT checkpoint.


In [6]:
EXPORT_ROOT = WORK_CACHE / 'chatquant_exports'
INT8_LINEAR_DIR = EXPORT_ROOT / f'{MODEL_TAG}_int8_linear'
FP16_COPY_DIR = EXPORT_ROOT / f'{MODEL_TAG}_fp16_copy'
OVERWRITE_QUANT_EXPORTS = True

if OVERWRITE_QUANT_EXPORTS:
    for path in [INT8_LINEAR_DIR, FP16_COPY_DIR]:
        shutil.rmtree(path, ignore_errors=True)

for path in [INT8_LINEAR_DIR, FP16_COPY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

run_cmd([
    sys.executable, '-m', 'scripts.chat_quantize',
    '--source', 'sft',
    '--model-tag', MODEL_TAG,
    '--step', str(SFT_STEP),
    '--method', 'int8_linear',
    '--output-dir', INT8_LINEAR_DIR,
])
run_cmd([
    sys.executable, '-m', 'scripts.chat_quantize',
    '--source', 'sft',
    '--model-tag', MODEL_TAG,
    '--step', str(SFT_STEP),
    '--method', 'fp16_copy',
    '--output-dir', FP16_COPY_DIR,
])

INT8_LINEAR_ARTIFACT = latest_quant_artifact(INT8_LINEAR_DIR)
FP16_COPY_ARTIFACT = latest_quant_artifact(FP16_COPY_DIR)
print('INT8 artifact:', INT8_LINEAR_ARTIFACT)
print('FP16 artifact:', FP16_COPY_ARTIFACT)


\n$ /usr/bin/python3 -m scripts.chat_quantize --source sft --model-tag d8_kaggle --step 4999 --method int8_linear --output-dir /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear
Exported quantized artifact to /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/quant_004999.pt
Metadata written to /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/meta_004999.json
Compression stats: original=503,317,416 bytes | exported=377,487,864 bytes | ratio=1.33x | quantized_tensors=54/63
\n$ /usr/bin/python3 -m scripts.chat_quantize --source sft --model-tag d8_kaggle --step 4999 --method fp16_copy --output-dir /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_fp16_copy
Exported quantized artifact to /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_fp16_copy/quant_004999.pt
Metadata written to /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_fp16_copy/meta_004999.json
Compression stats: original=503,317,416 bytes

## Export AWQ INT4 Artifact

Export an AWQ-style INT4 artifact using GSM8K calibration prompts.


In [7]:
RUN_AWQ_EXPORT = True
AWQ_CALIBRATION_EXAMPLES = 128
AWQ_MAX_CALIBRATION_TOKENS = 512
AWQ_ALPHA = 0.5
AWQ_INT4_DIR = EXPORT_ROOT / f'{MODEL_TAG}_awq_int4'

if RUN_AWQ_EXPORT:
    if OVERWRITE_QUANT_EXPORTS:
        shutil.rmtree(AWQ_INT4_DIR, ignore_errors=True)
    AWQ_INT4_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd([
        sys.executable, '-m', 'scripts.chat_quant_awq',
        '--source', 'sft',
        '--model-tag', MODEL_TAG,
        '--step', str(SFT_STEP),
        '--calibration-source', 'gsm8k',
        '--calibration-examples', str(AWQ_CALIBRATION_EXAMPLES),
        '--max-calibration-tokens', str(AWQ_MAX_CALIBRATION_TOKENS),
        '--alpha', str(AWQ_ALPHA),
        '--output-dir', AWQ_INT4_DIR,
    ])
    AWQ_INT4_ARTIFACT = latest_quant_artifact(AWQ_INT4_DIR)
    print('AWQ artifact:', AWQ_INT4_ARTIFACT)
else:
    AWQ_INT4_ARTIFACT = None
    print('Skipping AWQ export')


\n$ /usr/bin/python3 -m scripts.chat_quant_awq --source sft --model-tag d8_kaggle --step 4999 --calibration-source gsm8k --calibration-examples 128 --max-calibration-tokens 512 --alpha 0.5 --output-dir /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_awq_int4


2026-06-14 05:11:12,089 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:11:12,090 - datasets - INFO - JAX version 0.7.2 available.


Loaded checkpoint: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle step 4999


2026-06-14 05:11:14,114 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:11:14,114 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-14 05:11:14,129 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:11:14,145 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:11:14,209 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:11:14,319 - httpx - INFO - HTTP Request: HEAD htt

Collected 128 calibration prompts
Gathered activation stats for 54 linear modules
Exported AWQ-style INT4 artifact to /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_awq_int4/quant_004999.pt
Metadata written to /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_awq_int4/meta_004999.json
Compression stats: original=503,317,416 bytes | exported=356,944,408 bytes | ratio=1.41x | quantized_tensors=54/63
AWQ artifact: /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_awq_int4/quant_004999.pt


## Inspect Quant Artifacts

Check artifact metadata and sizes before evaluation or saving.


In [8]:
QUANT_ARTIFACTS = {
    'int8_linear': INT8_LINEAR_ARTIFACT,
    'fp16_copy': FP16_COPY_ARTIFACT,
}
if AWQ_INT4_ARTIFACT is not None:
    QUANT_ARTIFACTS['awq_int4'] = AWQ_INT4_ARTIFACT

for name, artifact_path in QUANT_ARTIFACTS.items():
    assert Path(artifact_path).exists(), f'Missing artifact: {artifact_path}'
    data = torch.load(artifact_path, map_location='cpu')
    print()
    print(name, artifact_path)
    print('quant_method:', data.get('quant_method'))
    print('source_model_tag:', data.get('source_model_tag'))
    print('source_step:', data.get('source_step'))
    print('artifact_format_version:', data.get('artifact_format_version'))
    print('size bytes:', Path(artifact_path).stat().st_size)

int8_data = torch.load(INT8_LINEAR_ARTIFACT, map_location='cpu')
fp16_data = torch.load(FP16_COPY_ARTIFACT, map_location='cpu')
assert int8_data['quant_method'] == 'int8_linear'
assert int8_data['quantize_embeddings'] is False
assert fp16_data['quant_method'] == 'fp16_copy'
assert fp16_data['quantize_embeddings'] is True
if AWQ_INT4_ARTIFACT is not None:
    awq_data = torch.load(AWQ_INT4_ARTIFACT, map_location='cpu')
    assert awq_data['quant_method'] == 'awq_int4'

subprocess.run(['du', '-sh', str(EXPORT_ROOT)], check=False)



int8_linear /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/quant_004999.pt
quant_method: int8_linear
source_model_tag: d8_kaggle
source_step: 4999
artifact_format_version: 2
size bytes: 377526204

fp16_copy /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_fp16_copy/quant_004999.pt
quant_method: fp16_copy
source_model_tag: d8_kaggle
source_step: 4999
artifact_format_version: 2
size bytes: 251683391

awq_int4 /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_awq_int4/quant_004999.pt
quant_method: awq_int4
source_model_tag: d8_kaggle
source_step: 4999
artifact_format_version: 2
size bytes: 356998174
941M	/kaggle/working/nanochat_cache/chatquant_exports


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache/chatquant_exports'], returncode=0)

## Quantized Evaluation

Evaluate the exported quant artifacts on a fixed GSM8K slice.


In [9]:
RUN_QUANT_EVAL = True
EVAL_TASK_NAME = 'GSM8K'
EVAL_MAX_PROBLEMS = 100
EVAL_MAX_NEW_TOKENS = 256
EVAL_BATCH_SIZE = 2

if RUN_QUANT_EVAL:
    for name, artifact_path in QUANT_ARTIFACTS.items():
        print('\\nEvaluating', name, artifact_path)
        run_cmd([
            sys.executable, '-m', 'scripts.chat_quant_eval',
            '--source', 'sft',
            '--model-tag', MODEL_TAG,
            '--step', str(SFT_STEP),
            '--quant-artifact', artifact_path,
            '--task-name', EVAL_TASK_NAME,
            '--max-problems', str(EVAL_MAX_PROBLEMS),
            '--max-new-tokens', str(EVAL_MAX_NEW_TOKENS),
            '--num-samples', '1',
            '--batch-size', str(EVAL_BATCH_SIZE),
        ])
else:
    print('Skipping quantized eval')


\nEvaluating int8_linear /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/quant_004999.pt
\n$ /usr/bin/python3 -m scripts.chat_quant_eval --source sft --model-tag d8_kaggle --step 4999 --quant-artifact /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/quant_004999.pt --task-name GSM8K --max-problems 100 --max-new-tokens 256 --num-samples 1 --batch-size 2
Autodetected device type: cuda


2026-06-14 05:11:35,339 - nanochat.common - INFO - Distributed world size: 1
2026-06-14 05:11:35,339 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle with step 4999
2026-06-14 05:11:36,022 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 1024, 'vocab_size': 32768, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'L'}
2026-06-14 05:11:37,041 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:11:37,042 - datasets - INFO - JAX version 0.7.2 available.


Evaluating baseline:sft


2026-06-14 05:11:37,795 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:11:37,796 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-14 05:11:37,810 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:11:37,869 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:11:37,968 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:11:38,034 - httpx - INFO - HTTP Request: GET https://huggin

Rank 0 | 0/100 (0.00%)
Final: 0/100 (0.00%)
baseline:sft | GSM8K: 0.00%
Evaluating quant:int8_linear


...

Evaluating baseline:sft


2026-06-14 05:22:09,881 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:22:09,881 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-14 05:22:09,897 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:22:09,960 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:22:10,078 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:22:10,152 - httpx - INFO - HTTP Request: GET https://huggin

Rank 0 | 0/100 (0.00%)
Final: 0/100 (0.00%)
baseline:sft | GSM8K: 0.00%
Evaluating quant:awq_int4


2026-06-14 05:24:42,392 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:24:42,408 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:24:42,469 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:24:42,595 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:24:42,658 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-06-14 05:24:42,783 - httpx - INFO - HTTP Request: GET https://datasets-serv

Rank 0 | 0/100 (0.00%)
Final: 0/100 (0.00%)
quant:awq_int4 | GSM8K: 0.00%

Model          | GSM8K | Mean 
---------------+-------+------
baseline:sft   | 0.00% | 0.00%
quant:awq_int4 | 0.00% | 0.00%


## Runtime-Quantized Eval

Exercise the `--runtime-quantized` path on the INT8 artifact.


In [10]:
RUN_RUNTIME_QUANTIZED_EVAL = True
RUNTIME_EVAL_MAX_PROBLEMS = 20

if RUN_RUNTIME_QUANTIZED_EVAL:
    run_cmd([
        sys.executable, '-m', 'scripts.chat_quant_eval',
        '--quant-artifact', INT8_LINEAR_ARTIFACT,
        '--runtime-quantized',
        '--task-name', 'GSM8K',
        '--max-problems', str(RUNTIME_EVAL_MAX_PROBLEMS),
        '--max-new-tokens', '128',
        '--num-samples', '1',
        '--batch-size', '1',
    ])
else:
    print('Skipping runtime-quantized eval')


\n$ /usr/bin/python3 -m scripts.chat_quant_eval --quant-artifact /kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/quant_004999.pt --runtime-quantized --task-name GSM8K --max-problems 20 --max-new-tokens 128 --num-samples 1 --batch-size 1
Autodetected device type: cuda


2026-06-14 05:27:33,595 - nanochat.common - INFO - Distributed world size: 1
2026-06-14 05:27:34,942 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:27:34,943 - datasets - INFO - JAX version 0.7.2 available.


Evaluating quant:int8_linear


2026-06-14 05:27:35,656 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:27:35,657 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-14 05:27:35,672 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:27:35,744 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:27:35,862 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:27:35,975 - httpx - INFO - HTTP Request: GET https://huggin

Rank 0 | 0/20 (0.00%)
Final: 0/20 (0.00%)
quant:int8_linear | GSM8K: 0.00%

Model             | GSM8K | Mean 
------------------+-------+------
quant:int8_linear | 0.00% | 0.00%


## Output Manifest

Write a compact manifest of the artifacts produced by this notebook.


In [11]:
manifest = {
    'model_tag': MODEL_TAG,
    'sft_checkpoint_dir': str(SFT_DIR),
    'sft_step': SFT_STEP,
    'export_root': str(EXPORT_ROOT),
    'quant_artifacts': {name: str(path) for name, path in QUANT_ARTIFACTS.items()},
    'artifact_sizes_bytes': {
        name: Path(path).stat().st_size
        for name, path in QUANT_ARTIFACTS.items()
        if Path(path).exists()
    },
    'tested_scripts': quant_scripts,
    'eval': {
        'task_name': EVAL_TASK_NAME,
        'max_problems': EVAL_MAX_PROBLEMS,
        'max_new_tokens': EVAL_MAX_NEW_TOKENS,
    },
}
manifest_path = Path('/kaggle/working/nanochat_d8_quant_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())

print('Artifact sizes:')
for name, path in QUANT_ARTIFACTS.items():
    print(name, path, Path(path).stat().st_size, 'bytes')
subprocess.run(['du', '-sh', str(EXPORT_ROOT)], check=False)


/kaggle/working/nanochat_d8_quant_manifest.json
{
  "model_tag": "d8_kaggle",
  "sft_checkpoint_dir": "/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle",
  "sft_step": 4999,
  "export_root": "/kaggle/working/nanochat_cache/chatquant_exports",
  "quant_artifacts": {
    "int8_linear": "/kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_int8_linear/quant_004999.pt",
    "fp16_copy": "/kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_fp16_copy/quant_004999.pt",
    "awq_int4": "/kaggle/working/nanochat_cache/chatquant_exports/d8_kaggle_awq_int4/quant_004999.pt"
  },
  "artifact_sizes_bytes": {
    "int8_linear": 377526204,
    "fp16_copy": 251683391,
    "awq_int4": 356998174
  },
  "tested_scripts": [
    "scripts/chat_quantize.py",
    "scripts/chat_quant_awq.py",
    "scripts/chat_quant_eval.py",
    "scripts/chat_web_quant.py"
  ],
  "eval": {
    "task_name": "GSM8K",
    "max_problems": 100,
    "max_new_tokens": 256
  }
}
Artifact sizes:
int8_linear /ka

CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache/chatquant_exports'], returncode=0)

In [12]:
# Optional: save the quant artifact cache as a Kaggle Dataset.
import json

QUANT_CACHE_DIR = Path('/kaggle/working/nanochat_d8_quant_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-quant-cache'
TITLE = 'nanochat d8 quant cache'
VERSION_MESSAGE = f'Save {MODEL_TAG} quant artifacts from SFT step {SFT_STEP}'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

required_paths = [EXPORT_ROOT, WORK_CACHE / 'tokenizer'] + [Path(path) for path in QUANT_ARTIFACTS.values()]
for required_path in required_paths:
    assert required_path.exists(), f'Missing required path: {required_path}'

if QUANT_CACHE_DIR.exists():
    shutil.rmtree(QUANT_CACHE_DIR)
QUANT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(EXPORT_ROOT, QUANT_CACHE_DIR / 'chatquant_exports')
shutil.copytree(WORK_CACHE / 'tokenizer', QUANT_CACHE_DIR / 'tokenizer')
if manifest_path.exists():
    shutil.copy2(manifest_path, QUANT_CACHE_DIR / manifest_path.name)

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = QUANT_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(QUANT_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(QUANT_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(QUANT_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Folder size:
942M	/kaggle/working/nanochat_d8_quant_cache
Running: kaggle datasets create -p /kaggle/working/nanochat_d8_quant_cache --dir-mode tar
Starting upload for file tokenizer.tar


100%|██████████| 540k/540k [00:00<00:00, 1.28MB/s]
  0%|          | 0.00/920 [00:00<?, ?B/s]

Upload successful: tokenizer.tar (540KB)
Starting upload for file nanochat_d8_quant_manifest.json


100%|██████████| 920/920 [00:00<00:00, 2.12kB/s]


Upload successful: nanochat_d8_quant_manifest.json (920B)
Starting upload for file chatquant_exports.tar


100%|██████████| 941M/941M [00:10<00:00, 96.3MB/s]


Upload successful: chatquant_exports.tar (941MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-quant-cache
Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-quant-cache


## Serve The Quantized Chat Model

Start a local NanoChat quantized web server from one exported artifact.


In [ ]:
import time
import requests

SERVE_ARTIFACT_NAME = 'int8_linear'  # choose 'int8_linear', 'fp16_copy', or 'awq_int4'
SERVE_RUNTIME_QUANTIZED = False
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'

assert SERVE_ARTIFACT_NAME in QUANT_ARTIFACTS, f'Unknown quant artifact: {SERVE_ARTIFACT_NAME}'
SERVE_QUANT_ARTIFACT = Path(QUANT_ARTIFACTS[SERVE_ARTIFACT_NAME])
assert SERVE_QUANT_ARTIFACT.exists(), f'Missing quant artifact: {SERVE_QUANT_ARTIFACT}'

try:
    if quant_server.poll() is None:
        print('Stopping existing NanoChat quantized server...')
        quant_server.terminate()
        quant_server.wait(timeout=10)
        print('Stopped existing quantized server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing quantized server cleanly:', exc)
    try:
        quant_server.kill()
        quant_server.wait(timeout=10)
        print('Killed existing quantized server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web_quant',
    '--quant-artifact', str(SERVE_QUANT_ARTIFACT),
    '--num-gpus', '1',
    '--host', '127.0.0.1',
    '--port', str(SERVER_PORT),
    '--max-tokens', '512',
]
if SERVE_RUNTIME_QUANTIZED:
    cmd.append('--runtime-quantized')

print('Starting NanoChat quantized server:')
print(' '.join(cmd))
quant_server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Quantized server process started with PID {quant_server.pid}.')

SERVER_READY = False
for _ in range(60):
    if quant_server.poll() is not None:
        raise RuntimeError(f'NanoChat quantized server exited early with code {quant_server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok and response.json().get('ready'):
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat quantized server is ready: {BASE_URL}')
else:
    print(f'NanoChat quantized server is still loading. Wait a bit, then use: {BASE_URL}')


In [ ]:
import json
import requests

BASE = globals().get('BASE_URL', 'http://127.0.0.1:8000')
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({'role': 'user', 'content': prompt})

    payload = {
        'messages': messages,
        'temperature': temperature,
        'top_k': top_k,
        'max_tokens': max_tokens,
    }

    print('Assistant:', end=' ', flush=True)
    answer = ''

    with requests.post(f'{BASE}/chat/completions', json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue

            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break

            token = data.get('token', '')
            answer += token
            print(token, end='', flush=True)

    print()
    messages.append({'role': 'assistant', 'content': answer})
    return answer

print(f'Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.')
while True:
    prompt = input('\nYou: ')
    command = prompt.strip().lower()
    if command in {'q', 'quit', 'exit'}:
        break
    if command == 'reset':
        messages.clear()
        print('Chat history cleared.')
        continue
    ask(prompt)

In [ ]:
# Stop the NanoChat quantized web server started by the serving cell.
import os
import subprocess
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('quant_server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat quantized server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat quantized server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat quantized server killed.')
        stopped_any = True
    else:
        print(f'NanoChat quantized server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `quant_server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web_quant' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web_quant process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web_quant process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat quantized server process found.')
